In [1]:
import pandas as pd
pd.set_option('display.max_columns',None)

In [2]:
df = pd.read_excel('task_2_data_ex.xlsx')

In [3]:
print(f'There is { len(df[df['produced_material_release_type']=='FIN'])}')

There is 120


In [4]:
#there is totaal 4 stages
df['produced_material_production_type'].nunique()

4

In [5]:
# there is only such number of unique fins
df[df['produced_material_release_type']=='FIN']['produced_material'].unique().__len__()


10

In [6]:
df[df['produced_material_release_type']=='FIN']['produced_material'].unique()


array([10000, 10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008,
       10009])

In [7]:
# i am filtering rows where includes type FIN
fin = df[df['produced_material_release_type'] == 'FIN']
# and cuting only cols i need 
fin = fin[['plant_id', 'year', 'produced_material', 'component_material']]


In [8]:
fin # there is 120 rows , and 10 produced material and one for each month so 10*12month=120 row

,plant_id,year,produced_material,component_material
0,RLT_10,2024,10000,50000
11,RLT_10,2024,10000,50000
22,RLT_10,2024,10000,50000
33,RLT_10,2024,10000,50000
44,RLT_10,2024,10000,50000
...,...,...,...,...
1265,RLT_14,2024,10009,50009
1276,RLT_14,2024,10009,50009
1287,RLT_14,2024,10009,50009
1298,RLT_14,2024,10009,50009


In [9]:
fin=fin.drop_duplicates() # we dont need duplicates 

In [10]:
# now we cut recipes table as well , and we dont need Duplcates in recipes as well so remove them
recipes = df[['plant_id','year','produced_material',
              'component_material','component_material_release_type']].drop_duplicates()


In [11]:
# so we can link them using merge 
step1=fin.merge(recipes,
          left_on=['plant_id','year','component_material'],
          right_on=['plant_id','year','produced_material'])

In [12]:
# lets rename them to see them with better more meaning ful name 
step1 = step1.rename(columns={
    'produced_material_x': 'fin_material_id',
    'produced_material_y': 'prod_material_id',
    'component_material_y': 'component_id'
}).drop(columns='component_material_x')


In [13]:
step1

,plant_id,year,fin_material_id,prod_material_id,component_id,component_material_release_type
0,RLT_10,2024,10000,50000,80070,PROD
1,RLT_10,2024,10000,50000,90000,ADD
2,RLT_10,2024,10000,50000,90001,ADD
3,RLT_10,2024,10001,50001,80071,PROD
4,RLT_10,2024,10001,50001,90005,ADD
5,RLT_10,2024,10001,50001,90006,ADD
6,RLT_14,2024,10002,50002,80072,PROD
7,RLT_14,2024,10002,50002,90010,ADD
8,RLT_14,2024,10002,50002,90011,ADD
9,RLT_14,2024,10003,50003,80073,PROD


In [14]:
step2 = step1[step1['component_material_release_type'] == 'PROD']


In [15]:
step2

,plant_id,year,fin_material_id,prod_material_id,component_id,component_material_release_type
0,RLT_10,2024,10000,50000,80070,PROD
3,RLT_10,2024,10001,50001,80071,PROD
6,RLT_14,2024,10002,50002,80072,PROD
9,RLT_14,2024,10003,50003,80073,PROD
12,RLT_14,2024,10004,50004,80074,PROD
15,RLT_14,2024,10005,50005,80075,PROD
18,RLT_14,2024,10006,50006,80076,PROD
21,RLT_16,2024,10007,50007,80077,PROD
24,RLT_16,2024,10008,50008,80078,PROD
27,RLT_14,2024,10009,50009,80079,PROD


In [16]:
step2 = step2[['plant_id','year','fin_material_id','component_id']]
step2 = step2.rename(columns={
    'component_id':'prod_material_id'
})

In [17]:
recipes

,plant_id,year,produced_material,component_material,component_material_release_type
0,RLT_10,2024,10000,50000,PROD
1,RLT_10,2024,50000,80070,PROD
2,RLT_10,2024,50000,90000,ADD
3,RLT_10,2024,50000,90001,ADD
4,RLT_10,2024,80070,80010,PROD
...,...,...,...,...,...
1194,RLT_14,2024,80079,90048,ADD
1195,RLT_14,2024,80019,80009,PROD
1196,RLT_14,2024,80019,90049,ADD
1197,RLT_14,2024,80009,70009,RM


In [18]:
step2.merge(recipes,
            left_on=['plant_id','year','prod_material_id'],
            right_on=['plant_id','year','produced_material'])


,plant_id,year,fin_material_id,prod_material_id,produced_material,component_material,component_material_release_type
0,RLT_10,2024,10000,80070,80070,80010,PROD
1,RLT_10,2024,10000,80070,80070,90002,ADD
2,RLT_10,2024,10000,80070,80070,90003,ADD
3,RLT_10,2024,10001,80071,80071,80011,PROD
4,RLT_10,2024,10001,80071,80071,90007,ADD
5,RLT_10,2024,10001,80071,80071,90008,ADD
6,RLT_14,2024,10002,80072,80072,80012,PROD
7,RLT_14,2024,10002,80072,80072,90012,ADD
8,RLT_14,2024,10002,80072,80072,90013,ADD
9,RLT_14,2024,10003,80073,80073,80013,PROD


# ANSWER WITH FUNCTION 



In [20]:
def next_level(current:pd.DataFrame,recipes:pd.DataFrame):
    prod = current[current['component_material_release_type']=='PROD']
    prod = prod[['plant_id','year','fin_material_id','component_id']]
    prod = prod.rename(columns={'component_id': 'prod_material_id'})
    result = prod.merge(recipes,
                        left_on=['plant_id', 'year', 'prod_material_id'],
                        right_on=['plant_id', 'year', 'produced_material'])
    return result.drop(columns='produced_material').rename(columns={'component_material': 'component_id'})


In [21]:
step2 = next_level(step1,recipes)

In [22]:
step3 = next_level(step2,recipes)
step4 = next_level(step3,recipes)

In [23]:
step4

,plant_id,year,fin_material_id,prod_material_id,component_id,component_material_release_type
0,RLT_10,2024,10000,80000,70000,RM
1,RLT_10,2024,10000,80000,90005,ADD
2,RLT_10,2024,10001,80001,70001,RM
3,RLT_10,2024,10001,80001,90010,ADD
4,RLT_14,2024,10002,80002,70002,RM
5,RLT_14,2024,10002,80002,90015,ADD
6,RLT_14,2024,10003,80003,70003,RM
7,RLT_14,2024,10003,80003,90020,ADD
8,RLT_14,2024,10004,80004,70004,RM
9,RLT_14,2024,10004,80004,90025,ADD


In [24]:
full = pd.concat([step1,step2,step3,step4],ignore_index=True)
full.shape

(100, 6)

In [25]:
full

,plant_id,year,fin_material_id,prod_material_id,component_id,component_material_release_type
0,RLT_10,2024,10000,50000,80070,PROD
1,RLT_10,2024,10000,50000,90000,ADD
2,RLT_10,2024,10000,50000,90001,ADD
3,RLT_10,2024,10001,50001,80071,PROD
4,RLT_10,2024,10001,50001,90005,ADD
...,...,...,...,...,...,...
95,RLT_16,2024,10007,80007,90040,ADD
96,RLT_16,2024,10008,80008,70008,RM
97,RLT_16,2024,10008,80008,90045,ADD
98,RLT_14,2024,10009,80009,70009,RM
